# Técnicas de Diseño y Validación de Modelos · Universidad Blas Pascal
## Preparación Examen Final · Banco 3
## Pipeline completo: de la formulación a la evaluación final (Módulos 1 a 5)

<div style="display:flex; justify-content:flex-start; margin: 8px 0 18px 0;">
  <a href="https://colab.research.google.com/github/GabrielaOjcius/Laboratorios-Tecnicas-Diseno-Validacion-Modelos/blob/main/Preparacion%20Examen%20Final/Banco3_Notebook.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/>
  </a>
  <span style="font-size: 0.98em;">En Colab, hac&eacute; una copia en tu Drive antes de editar.</span>
</div>

---

### Objetivo
Practicar los conceptos evaluados en la consigna del **Banco 3** del examen final: el ciclo completo de modelado, la partición correcta cuando hay múltiples registros por cliente, la elección de métricas ante desbalance, y la construcción de un pipeline con búsqueda de hiperparámetros sin fuga de datos.

### Alineación con el examen final
Corresponde a la consigna del **Banco 3** (Módulos 1 a 5, caso integrador) del examen final de Técnicas de Diseño y Validación de Modelos.

**Duración estimada:** 60 minutos

### Instrucciones
1. Ejecutar las celdas en orden (Entorno de ejecución → Ejecutar todo).
2. Completar las celdas de respuesta marcadas con `# ✏️ RESPONDA AQUÍ`.
3. No es necesario modificar el código, solo ejecutarlo y responder.

In [1]:
# ─── B.1. Datos de cancelación de clientes con varios registros por cliente ──
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    train_test_split, GroupShuffleSplit, GroupKFold, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, recall_score, roc_auc_score
from sklearn.datasets import make_classification

SEED = 42
rng = np.random.default_rng(SEED)

n_clientes = 500
registros_por_cliente = rng.integers(2, 6, size=n_clientes)  # cada cliente aparece varias veces
n_total = registros_por_cliente.sum()

X, y = make_classification(
    n_samples=n_total, n_features=6, n_informative=4,
    weights=[0.96, 0.04], class_sep=1.0, random_state=SEED  # 4 % de cancelaciones
)
client_id = np.repeat(np.arange(n_clientes), registros_por_cliente)

df = pd.DataFrame(X, columns=[f'feat_{i}' for i in range(X.shape[1])])
df['cliente_id'] = client_id
df['cancela'] = y

print('Registros totales:', len(df), '| Clientes únicos:', df['cliente_id'].nunique())
print('Prevalencia de cancelación: %.4f' % df['cancela'].mean())
df.head()

Registros totales: 1747 | Clientes únicos: 500
Prevalencia de cancelación: 0.0458


,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,cliente_id,cancela
0,0.302133,2.154328,-1.665450,0.367319,1.246158,-0.991275,0,0
1,-0.231210,-0.161478,0.003081,-1.039943,-1.335848,0.853105,0,0
2,-1.798382,-1.855769,0.074221,1.143638,-0.976250,-0.815865,1,0
3,-1.138569,1.613582,-2.605146,1.983602,1.349925,-2.568181,1,1
4,-3.064373,-1.159828,0.107027,-0.387565,-2.308989,-0.156912,1,0


In [2]:
# ─── B.2. Partición aleatoria por fila vs. partición por cliente ────────────
X_ = df.drop(columns=['cliente_id', 'cancela']).values
y_ = df['cancela'].values
grupos = df['cliente_id'].values

def evaluar_particion(nombre, idx_train, idx_test):
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(random_state=SEED))])
    pipe.fit(X_[idx_train], y_[idx_train])
    proba = pipe.predict_proba(X_[idx_test])[:, 1]
    solapamiento = len(set(grupos[idx_train]) & set(grupos[idx_test]))
    print(f'{nombre}: clientes en train Y test = {solapamiento} | AP test = {average_precision_score(y_[idx_test], proba):.4f}')

# Partición aleatoria por fila (ignora que un mismo cliente puede caer en train y test)
idx_train_rand, idx_test_rand = train_test_split(
    np.arange(len(df)), test_size=0.25, stratify=y_, random_state=SEED
)
evaluar_particion('Partición aleatoria por fila', idx_train_rand, idx_test_rand)

# Partición respetando al cliente como unidad de generalización
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
idx_train_grp, idx_test_grp = next(gss.split(X_, y_, groups=grupos))
evaluar_particion('Partición por cliente (GroupShuffleSplit)', idx_train_grp, idx_test_grp)

Partición aleatoria por fila: clientes en train Y test = 282 | AP test = 0.2161
Partición por cliente (GroupShuffleSplit): clientes en train Y test = 0 | AP test = 0.2894


### Pregunta 1 — Ciclo de modelado y partición correcta (Módulos 1 y 2)
Describa brevemente las etapas del ciclo de modelado predictivo, desde la formulación del problema hasta la puesta en operación. Luego, compare el solapamiento de clientes entre train y test en ambas particiones: ¿por qué la partición aleatoria por fila puede producir una estimación optimista?

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

In [3]:
# ─── B.3. Pipeline evaluado con GroupKFold (sin fuga de datos) ──────────────
X_train_g, X_test_g = X_[idx_train_grp], X_[idx_test_grp]
y_train_g, y_test_g = y_[idx_train_grp], y_[idx_test_grp]
grupos_train_g = grupos[idx_train_grp]

pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(random_state=SEED))])
gkf = GroupKFold(n_splits=5)

aps, recalls = [], []
for tr_idx, val_idx in gkf.split(X_train_g, y_train_g, groups=grupos_train_g):
    pipe.fit(X_train_g[tr_idx], y_train_g[tr_idx])
    proba_val = pipe.predict_proba(X_train_g[val_idx])[:, 1]
    aps.append(average_precision_score(y_train_g[val_idx], proba_val))
    recalls.append(recall_score(y_train_g[val_idx], (proba_val >= 0.5).astype(int), zero_division=0))

print('GroupKFold (5 folds, sin fuga entre clientes):')
print('  Average Precision: %.4f ± %.4f' % (np.mean(aps), np.std(aps)))
print('  Recall (umbral 0.5): %.4f ± %.4f' % (np.mean(recalls), np.std(recalls)))

GroupKFold (5 folds, sin fuga entre clientes):
  Average Precision: 0.3070 ± 0.0791
  Recall (umbral 0.5): 0.0000 ± 0.0000


### Pregunta 2 — Elección de métrica (Módulo 2)
Dado que las cancelaciones representan solo el 4 % de los casos, ¿qué métrica priorizaría para detectar cancelaciones: accuracy, precision, recall, F1 o Average Precision? Justifique con los valores obtenidos.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

In [4]:
# ─── B.4. Búsqueda de hiperparámetros dentro del pipeline (test aislado) ────
param_grid = {'clf__C': [0.01, 0.1, 1.0, 10.0]}

grid = GridSearchCV(pipe, param_grid, cv=GroupKFold(n_splits=5), scoring='average_precision')
grid.fit(X_train_g, y_train_g, groups=grupos_train_g)

print('Mejor C encontrado (solo con datos de train, vía GroupKFold):', grid.best_params_)
print('Mejor Average Precision en validación cruzada: %.4f' % grid.best_score_)

# Evaluación final ÚNICA sobre el test separado por cliente, nunca usado hasta ahora
mejor_pipe = grid.best_estimator_
proba_test = mejor_pipe.predict_proba(X_test_g)[:, 1]
print('\nEvaluación final sobre test (clientes nunca vistos):')
print('  Average Precision test: %.4f' % average_precision_score(y_test_g, proba_test))
print('  ROC-AUC test:           %.4f' % roc_auc_score(y_test_g, proba_test))

Mejor C encontrado (solo con datos de train, vía GroupKFold): {'clf__C': 0.01}
Mejor Average Precision en validación cruzada: 0.3427

Evaluación final sobre test (clientes nunca vistos):
  Average Precision test: 0.3315
  ROC-AUC test:           0.5341


### Pregunta 3 — Pipeline completo sin fuga de datos (Módulos 4 y 5)
Explique cómo se organizó este pipeline para que el escalado (`StandardScaler`) y la búsqueda de `C` se ajustaran únicamente con datos de entrenamiento, dejando el conjunto de test reservado solo para la evaluación final. ¿Qué hubiera pasado si se ajustaba el `StandardScaler` con todos los datos antes de dividir en train/test?

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

---
### Checklist de autoevaluación — Banco 3
- [ ] Puedo describir el ciclo completo de modelado predictivo.
- [ ] Puedo explicar por qué la unidad de partición (cliente) importa para evitar fuga de datos.
- [ ] Puedo justificar la métrica adecuada ante clases desbalanceadas.
- [ ] Puedo armar un pipeline (escalado + modelo) y ajustar hiperparámetros sin tocar el test hasta la evaluación final.